# Language detection, step by step

This is the exploration version of `src/data_model_pipeline`: the same steps,
run one cell at a time so you can look at the data as you go.

Takes several minutes and a few GB of memory on the full 2.7M rows.
To experiment faster, add `df = df.sample(200_000, random_state=42)` after loading.


In [ ]:
# Import necessary libraries
import os
import re
import warnings

warnings.simplefilter("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder


## Data Ingestion


In [ ]:
df = pd.read_csv("../data/raw/new_merged_languages.csv")


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.duplicated().sum())


In [ ]:
# Drop unwanted columns
df = df.drop(columns=['new_id', 'old_id', 'data_source', 'label'], errors='ignore')

# Check remaining columns
df.head()


In [ ]:
df.shape

In [ ]:
df['language'].value_counts()


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['language'],  # keeps language distribution same across splits
    random_state=42
)

In [ ]:
# Then, split temp into validation (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # half of 20% = 10%
    stratify=temp_df['language'],
    random_state=42
)

In [ ]:
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
df

## DATA PREPROCESSING

In [ ]:
# 2️⃣ Basic cleaning (remove URLs, digits, special chars, lowercase)
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+|@\S+", "", text)  # remove URLs and mentions
    text = re.sub(r"\d+", "", text)                  # remove numbers
    text = re.sub(r"[^\w\s]", "", text)              # remove punctuation
    return text.lower().strip()



In [ ]:
train_df_clean= train_df["prompt"].apply(clean_text)

In [ ]:
X_train_pd= train_df_clean

In [ ]:
X_train= train_df_clean.values

In [ ]:
val_df_clean = val_df["prompt"].apply(clean_text)

In [ ]:
X_val= val_df_clean.values

In [ ]:
test_df_clean = test_df["prompt"].apply(clean_text)

In [ ]:
X_test= test_df_clean.values

In [ ]:
train_df_clean.shape, val_df_clean.shape, test_df_clean.shape

In [ ]:
train_df_clean.head(20)

In [ ]:
y_val=val_df["language"].values

In [ ]:
y_val

In [ ]:
y_test=test_df["language"].values

In [ ]:
y_train=train_df["language"].values


In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000,      # Control memory/performance
    ngram_range=(1, 3),      # Capture character/word patterns
    analyzer='char_wb',      # Highly recommended for language ID! Uses characters inside word boundaries.
    min_df=10                # Ignore very rare tokens
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val) # Important: use transform!
# Save the vectorizer with joblib for production

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train_encoded)

In [ ]:
# Load the artifacts saved by the pipeline in src/data_model_pipeline.
# Run the pipeline first (see the README).
models_dir = os.path.join("..", "models")

vectorizer = joblib.load(os.path.join(models_dir, "vectorizer.joblib"))
label_encoder = joblib.load(os.path.join(models_dir, "label_encoder.joblib"))
model = joblib.load(os.path.join(models_dir, "language_detector.joblib"))

print(f"Loaded a model that knows {len(label_encoder.classes_)} languages")


In [ ]:


# Make predictions
def predict_language(text):
    # Transform text
    text_tfidf = vectorizer.transform([clean_text(text)])
    
    # Predict
    encoded_pred = model.predict(text_tfidf)
    
    # Convert back to language name
    language = label_encoder.inverse_transform(encoded_pred)
    return language[0]

# Test it
print(predict_language("I am going to scholls tomorrow"))  # Should output 'English'
print(predict_language("التعلم الآلي يحول عالم التكنولوجيا"))       # Should output 'Arabic'

In [ ]:
def test_language_detection(model, vectorizer, label_encoder):
    test_sentences = [
        # English
        "The quick brown fox jumps over the lazy dog.",
        "Hello, how are you doing today?",
        
        # Spanish
        "El rápido zorro marrón salta sobre el perro perezoso.",
        "Hola, ¿cómo estás hoy?",
        
        # French
        "Le rapide renard brun saute par-dessus le chien paresseux.",
        "Bonjour, comment allez-vous aujourd'hui ?",
        
        # German
        "Der schnelle braune Fuchs springt über den faulen Hund.",
        "Hallo, wie geht es dir heute?",
        
        # Hindi
        "तेज भूरी लोमड़ी आलसी कुत्ते के ऊपर कूदती है।",
        "नमस्ते, आप आज कैसे हैं?",
        
        # Chinese
        "快速的棕色狐狸跳过懒狗。",
        "你好，你今天怎么样？"
    ]
    
    expected_languages = [
        'English', 'English',
        'Spanish', 'Spanish', 
        'French', 'French',
        'German', 'German',
        'Hindi', 'Hindi',
        'Chinese', 'Chinese'
    ]
    
    correct = 0
    total = len(test_sentences)
    
    for i, sentence in enumerate(test_sentences):
        # Transform the sentence
        features = vectorizer.transform([clean_text(sentence)])
        
        # Predict
        prediction = model.predict(features)
        predicted_language = label_encoder.inverse_transform(prediction)[0]
        
        # Compare
        expected = expected_languages[i]
        is_correct = predicted_language == expected
        if is_correct:
            correct += 1
            
        print(f"Sentence: {sentence[:50]}...")
        print(f"Expected: {expected}, Predicted: {predicted_language}, Correct: {is_correct}")
        print("-" * 50)
    
    accuracy = (correct / total) * 100
    print(f"\nAccuracy: {correct}/{total} ({accuracy:.2f}%)")
    
    return accuracy

# Usage
accuracy = test_language_detection(model, vectorizer, label_encoder)

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val_tfidf)

# Calculate basic accuracy
basic_accuracy = accuracy_score(y_val_encoded, y_val_pred)
print(f"✅ Basic Accuracy: {basic_accuracy:.4f} ({basic_accuracy*100:.2f}%)")

In [ ]:
def quick_model_summary(model, X_val_tfidf, y_val_encoded, label_encoder):
    y_val_pred = model.predict(X_val_tfidf)
    accuracy = accuracy_score(y_val_encoded, y_val_pred)
    
    print("🚀 MODEL PERFORMANCE SUMMARY")
    print("=" * 40)
    print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Number of Languages: {len(label_encoder.classes_)}")
    print(f"Validation Samples: {len(y_val_encoded)}")
    
    # Show sample predictions
    print("\n🔍 Sample Predictions:")
    sample_indices = np.random.choice(len(y_val_encoded), 5, replace=False)
    for idx in sample_indices:
        actual = label_encoder.inverse_transform([y_val_encoded[idx]])[0]
        predicted = label_encoder.inverse_transform([y_val_pred[idx]])[0]
        correct = "✅" if actual == predicted else "❌"
        print(f"   {correct} Actual: {actual:15} Predicted: {predicted:15}")

# Quick summary
quick_model_summary(model, X_val_tfidf, y_val_encoded, label_encoder)

In [ ]:
def evaluate_language_model(model, X_val_tfidf, y_val_encoded, label_encoder):
    # Predictions
    y_val_pred = model.predict(X_val_tfidf)
    
    # Convert back to language names for readability
    y_val_languages = label_encoder.inverse_transform(y_val_encoded)
    y_pred_languages = label_encoder.inverse_transform(y_val_pred)
    
    # 1. Overall Accuracy
    accuracy = accuracy_score(y_val_encoded, y_val_pred)
    print(f"🎯 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print()
    
    # 2. Detailed Classification Report
    print("📊 Detailed Classification Report:")
    print(classification_report(y_val_languages, y_pred_languages, zero_division=0))
    
    # 3. Per-language accuracy
    print("🌍 Per-Language Accuracy (Top 20):")
    cm = confusion_matrix(y_val_encoded, y_val_pred)
    language_accuracies = cm.diagonal() / cm.sum(axis=1)
    
    # Create accuracy per language
    language_results = []
    for i, lang in enumerate(label_encoder.classes_):
        lang_accuracy = language_accuracies[i]
        support = cm[i].sum()  # Number of samples for this language
        language_results.append((lang, lang_accuracy, support))
    
    # Sort by accuracy and show top 20
    language_results.sort(key=lambda x: x[1], reverse=True)
    for lang, acc, supp in language_results[:20]:
        print(f"   {lang:15} {acc:.4f} ({acc*100:.2f}%) - {supp} samples")
    
    # Show worst 5 languages
    print("\n⚠️  Languages Needing Improvement (Bottom 5):")
    for lang, acc, supp in language_results[-5:]:
        print(f"   {lang:15} {acc:.4f} ({acc*100:.2f}%) - {supp} samples")
    
    return y_val_pred, language_results

# Run evaluation
y_val_pred, language_results = evaluate_language_model(model, X_val_tfidf, y_val_encoded, label_encoder)

In [ ]:
def plot_top_languages_confusion_matrix(model, X_val_tfidf, y_val_encoded, label_encoder, top_n=15):
    # Get top N languages by sample count
    from collections import Counter
    language_counts = Counter(label_encoder.inverse_transform(y_val_encoded))
    top_languages = [lang for lang, count in language_counts.most_common(top_n)]
    
    # Filter validation set for top languages only
    top_language_indices = [i for i, lang in enumerate(label_encoder.classes_) if lang in top_languages]
    mask = np.isin(y_val_encoded, top_language_indices)
    
    X_val_top = X_val_tfidf[mask]
    y_val_top = y_val_encoded[mask]
    
    # Predict and create confusion matrix
    y_pred_top = model.predict(X_val_top)
    
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(y_val_top, y_pred_top, labels=top_language_indices)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=top_languages, 
                yticklabels=top_languages)
    plt.title(f'Confusion Matrix - Top {top_n} Languages')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Plot for top 15 languages (adjust based on your data)
plot_top_languages_confusion_matrix(model, X_val_tfidf, y_val_encoded, label_encoder, top_n=15)

In [ ]:
# Calculate metrics that handle class imbalance
precision = precision_score(y_val_encoded, y_val_pred, average='weighted')
recall = recall_score(y_val_encoded, y_val_pred, average='weighted')
f1 = f1_score(y_val_encoded, y_val_pred, average='weighted')

print(f"📈 Weighted Metrics (accounts for class imbalance):")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}") 
print(f"   F1-Score:  {f1:.4f}")